In [1]:
from hmmlearn.hmm import GaussianHMM
import numpy as np
import joblib
import utils.data as d



In [2]:
def zscore(df):
    out = df.copy()
    for col in HMM_FEATURES:
        mu = out[col].rolling(ZSCORE_WINDOW).mean()
        sigma = out[col].rolling(ZSCORE_WINDOW).std()
        out[f"{col}_z"] = (out[col] - mu) / sigma
    return out.dropna()

def train_hmm(df, n_restarts=10):
    X = df[[f"{c}_z" for c in HMM_FEATURES]].values
    best_model, best_score = None, -np.inf
    for i in range(n_restarts):
        model = GaussianHMM(
            n_components=2,
            covariance_type="full",
            n_iter=100,
            tol=1e-4,
            random_state=i,
        )
        model.fit(X)
        score = model.score(X)
        if score > best_score:
            best_model, best_score = model, score
    return best_model



In [5]:
HMM_FEATURES = ["es_rvol", "es_efficiency", "es_vol_ratio", "es_rel_volume"]
ZSCORE_WINDOW = 240

data = d.get_data()
data = zscore(data)

model = train_hmm(data)
joblib.dump(model, "models/cl_hmm.pkl")

Z_COLS = [f"{c}_z" for c in HMM_FEATURES]

means = data.groupby(model.predict(data[Z_COLS].values))[HMM_FEATURES].mean()
tradable_id = int(means["es_rvol"].idxmax())

posteriors = model.predict_proba(data[Z_COLS].values)
data["tradable_pct"] = posteriors[:, tradable_id]

print(f"LL: {model.score(data[Z_COLS].values):.2f}  |  tradable_id={tradable_id}\n")
print(data["tradable_pct"].describe())

LL: -28234.70  |  tradable_id=1

count    5.656000e+03
mean     3.391622e-01
std      4.577864e-01
min      7.922331e-66
25%      7.564123e-05
50%      9.484633e-04
75%      9.996810e-01
max      1.000000e+00
Name: tradable_pct, dtype: float64


In [4]:
HMM_FEATURES = ["cl_rvol", "cl_efficiency", "cl_vol_ratio", "cl_rel_volume"]
ZSCORE_WINDOW = 240

data = d.get_data()
data = zscore(data)

model = train_hmm(data)
joblib.dump(model, "models/cl_hmm.pkl")

Z_COLS = [f"{c}_z" for c in HMM_FEATURES]

means = data.groupby(model.predict(data[Z_COLS].values))[HMM_FEATURES].mean()
tradable_id = int(means["cl_efficiency"].idxmax())

posteriors = model.predict_proba(data[Z_COLS].values)
data["tradable_pct"] = posteriors[:, tradable_id]

print(f"LL: {model.score(data[Z_COLS].values):.2f}  |  tradable_id={tradable_id}\n")
print(data["tradable_pct"].describe())

LL: -29680.93  |  tradable_id=0

count    5.656000e+03
mean     4.702717e-01
std      4.717687e-01
min      1.129596e-84
25%      2.487687e-04
50%      2.340327e-01
75%      9.991585e-01
max      1.000000e+00
Name: tradable_pct, dtype: float64
